# News Topic Classifier Using BERT — AG News

## Objective
Fine-tune `bert-base-uncased` on the AG News dataset to classify news headlines into topic categories, evaluate with accuracy and F1-score, and prepare a lightweight Gradio demo for live interaction.

## Notebook Plan
1. Load and inspect the AG News dataset  
2. Visualize class balance and headline length  
3. Tokenize and preprocess text  
4. Fine-tune BERT with Hugging Face Transformers  
5. Evaluate using accuracy, F1-score, and confusion matrix  
6. Test on real headlines  
7. Save the model and provide a Gradio demo

In [ ]:
# Optional installs (uncomment if needed in your Kaggle environment)
# !pip -q install transformers datasets accelerate evaluate gradio

import os
import re
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch

from datasets import load_dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "bert-base-uncased"
OUTPUT_DIR = Path("/kaggle/working/ag_news_bert")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3
SAVE_TOTAL_LIMIT = 2

# Use a smaller subset for quick debugging if needed
MAX_TRAIN_SAMPLES = None  # e.g. 20000 for a fast run
MAX_TEST_SAMPLES = None   # e.g. 2000 for a fast run

print("Model:", MODEL_NAME)
print("Output dir:", OUTPUT_DIR)
print("Max length:", MAX_LENGTH)

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

dataset = load_dataset("ag_news")
print(dataset)

label_names = ["World", "Sports", "Business", "Sci/Tech"]
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

print("Label mapping:", id2label)

In [ ]:
# ============================================================
# QUICK INSPECTION
# ============================================================

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

print("\nTrain label distribution:")
display(train_df["label"].value_counts().sort_index().rename(index=id2label))

In [ ]:
# ============================================================
# EXPLORATORY DATA ANALYSIS
# ============================================================

plt.figure(figsize=(6, 4))
sns.countplot(x="label", data=train_df)
plt.xticks(ticks=[0, 1, 2, 3], labels=label_names)
plt.title("AG News Class Distribution")
plt.xlabel("Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

train_df["headline_len_words"] = train_df["text"].astype(str).str.split().apply(len)

plt.figure(figsize=(8, 4))
sns.histplot(train_df["headline_len_words"], bins=30, kde=True)
plt.title("Headline Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x=train_df["headline_len_words"])
plt.title("Headline Length Boxplot")
plt.xlabel("Number of Words")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# OPTIONAL SAMPLE VISUALIZATION OF HEADLINES
# ============================================================

sample_examples = train_df.sample(8, random_state=SEED)[["text", "label"]].copy()
sample_examples["category"] = sample_examples["label"].map(id2label)

display(sample_examples)

In [ ]:
# ============================================================
# TRAIN / VALIDATION SPLIT
# ============================================================

train_split, val_split = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df["label"]
)

if MAX_TRAIN_SAMPLES is not None and len(train_split) > MAX_TRAIN_SAMPLES:
    train_split = train_split.sample(n=MAX_TRAIN_SAMPLES, random_state=SEED)

if MAX_TEST_SAMPLES is not None and len(test_df) > MAX_TEST_SAMPLES:
    test_df = test_df.sample(n=MAX_TEST_SAMPLES, random_state=SEED)

train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_split.shape)
print("Validation:", val_split.shape)
print("Test:", test_df.shape)

print("\nTrain label counts:")
display(train_split["label"].value_counts().sort_index().rename(index=id2label))

In [ ]:
# ============================================================
# TOKENIZATION
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH
    )

# Convert pandas to Hugging Face datasets
from datasets import Dataset

train_hf = Dataset.from_pandas(train_split[["text", "label"]], preserve_index=False)
val_hf = Dataset.from_pandas(val_split[["text", "label"]], preserve_index=False)
test_hf = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

train_hf = train_hf.map(tokenize_batch, batched=True)
val_hf = val_hf.map(tokenize_batch, batched=True)
test_hf = test_hf.map(tokenize_batch, batched=True)

train_hf = train_hf.rename_column("label", "labels")
val_hf = val_hf.rename_column("label", "labels")
test_hf = test_hf.rename_column("label", "labels")

cols_to_keep = ["input_ids", "attention_mask", "labels"]
train_hf.set_format(type="torch", columns=cols_to_keep)
val_hf.set_format(type="torch", columns=cols_to_keep)
test_hf.set_format(type="torch", columns=cols_to_keep)

print(train_hf)
print(val_hf)
print(test_hf)

In [ ]:
# ============================================================
# MODEL DEFINITION
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()}
).to(DEVICE)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Model and data collator ready.")

In [ ]:
# ============================================================
# TRAINING ARGUMENTS
# ============================================================

import inspect
ta_params = inspect.signature(TrainingArguments.__init__).parameters

training_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

# Handle version differences in transformers
if "evaluation_strategy" in ta_params:
    training_kwargs["evaluation_strategy"] = "epoch"
else:
    training_kwargs["eval_strategy"] = "epoch"

if "save_strategy" in ta_params:
    training_kwargs["save_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

print("TrainingArguments ready.")

In [ ]:
# ============================================================
# METRICS
# ============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

print("Metric function ready.")

In [ ]:
# ============================================================
# TRAINING
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

train_result = trainer.train()
print("Training completed.")
print(train_result)

In [ ]:
# ============================================================
# EVALUATION ON TEST SET
# ============================================================

test_pred = trainer.predict(test_hf)
test_logits = test_pred.predictions
test_labels = test_pred.label_ids
test_preds = np.argmax(test_logits, axis=-1)

test_accuracy = accuracy_score(test_labels, test_preds)
test_f1_macro = f1_score(test_labels, test_preds, average="macro")
test_f1_weighted = f1_score(test_labels, test_preds, average="weighted")

print("Test Accuracy:", round(test_accuracy, 4))
print("Test F1 Macro:", round(test_f1_macro, 4))
print("Test F1 Weighted:", round(test_f1_weighted, 4))

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=label_names))

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.title("Confusion Matrix — AG News BERT Classifier")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ERROR ANALYSIS
# ============================================================

wrong_idx = np.where(test_preds != test_labels)[0]

print("Total misclassified samples:", len(wrong_idx))

if len(wrong_idx) > 0:
    sample_wrong = np.random.RandomState(SEED).choice(
        wrong_idx,
        size=min(15, len(wrong_idx)),
        replace=False
    )

    rows = []
    probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()

    for i in sample_wrong:
        true_lab = label_names[int(test_labels[i])]
        pred_lab = label_names[int(test_preds[i])]
        top3 = np.argsort(probs[i])[-3:][::-1]
        rows.append({
            "headline": test_df.iloc[i]["text"],
            "true": true_lab,
            "predicted": pred_lab,
            "top1_prob": float(probs[i][np.argmax(test_logits[i])]),
            "top3": ", ".join([f"{label_names[j]} ({probs[i][j]:.2f})" for j in top3])
        })

    display(pd.DataFrame(rows))

In [ ]:
# ============================================================
# REAL-WORLD TESTING
# ============================================================

def predict_headline(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(DEVICE)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    top3 = np.argsort(probs)[-3:][::-1]
    return [
        {"label": label_names[i], "probability": float(probs[i])}
        for i in top3
    ]

real_tests = [
    "Apple unveils next generation AI chips",
    "Manchester United wins dramatic final",
    "Stock markets rally after inflation report",
    "UN announces new peace agreement",
    "Scientists discover new exoplanet with water",
    "Government introduces new tax policy"
]

for headline in real_tests:
    preds = predict_headline(headline)
    print("\nHeadline:", headline)
    for item in preds:
        print(f"  {item['label']}: {item['probability']:.2%}")

In [ ]:
# ============================================================
# SAVE MODEL AND TOKENIZER
# ============================================================

SAVE_DIR = OUTPUT_DIR / "ag_news_bert_classifier"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(SAVE_DIR / "label_names.json", "w") as f:
    json.dump(label_names, f, indent=2)

print("Saved model to:", SAVE_DIR)

In [ ]:
# ============================================================
# GRADIO DEMO
# ============================================================

# Uncomment and run if gradio is available in your environment.
# import gradio as gr
#
# def gradio_predict(text):
#     results = predict_headline(text)
#     top1 = results[0]
#     return top1["label"], top1["probability"], results
#
# demo = gr.Interface(
#     fn=gradio_predict,
#     inputs=gr.Textbox(lines=3, placeholder="Enter a news headline..."),
#     outputs=[
#         gr.Textbox(label="Predicted Category"),
#         gr.Number(label="Confidence"),
#         gr.JSON(label="Top-3 Predictions")
#     ],
#     title="AG News Topic Classifier",
#     description="Fine-tuned BERT classifier for AG News headlines"
# )
#
# demo.launch()

## Final Insights
- AG News is a balanced four-class text classification benchmark.
- BERT is well suited because it learns contextual meaning from raw headlines.
- Accuracy and macro F1 are the key metrics for balanced multi-class classification.
- Saving the model and tokenizer makes the notebook deployment-ready.